# Stage 02 — Complex Comparison: Rips vs Alpha

Compares Vietoris-Rips and Alpha complexes on noisy shapes.
Visualizes geometric structure (Voronoi cells, Delaunay triangulation, Alpha balls)
and quantifies diagram similarity via bottleneck and Wasserstein distances.

**Three-layer module structure**

| Module | Responsibility |
|---|---|
| `tda.tda_data` | Pure computation — generate, compute, filter, stats, distances |
| `tda.tda_viz` | Atomic ax-based renderers — `render_*` functions |
| `tda.tda_figures` | GridSpec orchestrators — `plot_*` functions |

In [ ]:
import sys
sys.path.insert(0, '..')

from tda import (
    # data layer
    generate_point_cloud, compute_rips, compute_alpha, compute_both,
    filter_diagrams, h1_stats, auto_alpha_value, diagram_distances,
    # renderer layer
    render_point_cloud, render_persistence_diagram, render_barcode, render_landscape,
    render_comparison_table, render_matching,
    render_voronoi_delaunay, render_alpha_overlay, render_vr_overlay,
    # figure layer
    plot_complex_comparison, plot_geometric, plot_alpha_vr_comparison,
    plot_distance_comparison, plot_full_analysis, print_distance_table,
)
import matplotlib.pyplot as plt

## Layer 1 — `tda_data`: Computation

All functions return typed dataclasses — no 7-tuple unpacking.

| Function | Returns |
|---|---|
| `generate_point_cloud(shape, n, noise)` | `ndarray` — `'circle'` \| `'torus'` \| `'sphere'` |
| `compute_rips(pts, max_dim)` | `RipsResult` — `.dgms`, `.num_edges`, `.elapsed_ms` |
| `compute_alpha(pts, max_dim)` | `AlphaResult` — `.dgms`, `.num_simplices`, `.elapsed_ms` |
| `compute_both(shape, n, noise, max_dim)` | `BothResult` — `.pts`, `.rips`, `.alpha`, `.shape`, `.noise` |
| `filter_diagrams(dgms, threshold)` | filtered list; keeps ∞ bars, drops short finite bars |
| `h1_stats(dgms)` | `(bar_count, max_persistence)` |
| `auto_alpha_value(dgms_alpha)` | birth of top-persistence H1 bar (GUDHI r² units) |
| `diagram_distances(dgms_rips, dgms_alpha)` | `{dim: {bottleneck, wasserstein, bn_match, ws_match, d_rips, d_alpha}}` |

In [ ]:
# compute_both — generates a point cloud and runs Rips + Alpha in one call
result = compute_both('circle', n_points=100, noise=0.05)
print(f"pts shape: {result.pts.shape}  |  noise: {result.noise}")
print(f"Rips:  {result.rips.num_edges} edges,      {result.rips.elapsed_ms:.1f}ms")
print(f"Alpha: {result.alpha.num_simplices} simplices,  {result.alpha.elapsed_ms:.1f}ms")

# compute_rips / compute_alpha — use independently when you only need one
rips_only = compute_rips(result.pts, max_dim=1)
alpha_only = compute_alpha(result.pts, max_dim=1)
print(f"\nIndependent: Rips edges={rips_only.num_edges}, Alpha simplices={alpha_only.num_simplices}")

# filter_diagrams — unified filter that works on any diagram list
rips_filt = filter_diagrams(result.rips.dgms, threshold=0.05)
print(f"\nRips H1 bars after filter(0.05): {len(result.rips.dgms[1])} → {len(rips_filt[1])}")

# h1_stats — inspect the H1 diagram
count, max_pers = h1_stats(result.rips.dgms)
print(f"h1_stats: {count} finite bars, max persistence = {max_pers:.4f}")

# auto_alpha_value — picks the alpha parameter from the dominant H1 bar (GUDHI r² units)
av = auto_alpha_value(result.alpha.dgms)
print(f"\nauto_alpha_value: {av:.4f}  (r² units → r≈{av**0.5:.4f})")

# diagram_distances — computes bottleneck + Wasserstein per homology dimension
distances = diagram_distances(result.rips.dgms, result.alpha.dgms)

# print_distance_table — stdout summary of distances
print("\nRips vs Alpha — Circle  |  noise=0.05")
print_distance_table(distances)

## Layer 2 — `tda_viz`: Renderers

All renderers take `(data, ax, **kwargs)` and return `None`. No `plt.show()`, no GridSpec.

**Persistence representations**

| Function | Key kwargs |
|---|---|
| `render_point_cloud(pts, ax)` | `title` |
| `render_persistence_diagram(dgms, ax)` | `title` |
| `render_barcode(dgms, ax)` | `title`, `dims` |
| `render_landscape(dgms, ax)` | `title`, `dims`, `n_landscapes` |
| `render_comparison_table(rips, alpha, ax)` | takes `RipsResult` + `AlphaResult` |
| `render_matching(d_r, d_a, matching, ax)` | `title` |

**Geometric renderers (2D only)**

| Function | Key kwargs |
|---|---|
| `render_voronoi_delaunay(pts, ax)` | `circumsphere`, `seed` |
| `render_alpha_overlay(pts, alpha_value, ax)` | `circumcenter`, `seed` |
| `render_vr_overlay(pts, radius, ax)` | — |

In [ ]:
result = compute_both('circle', n_points=60, noise=0.05, max_dim=1)
av = auto_alpha_value(result.alpha.dgms)
distances = diagram_distances(result.rips.dgms, result.alpha.dgms)
d1 = distances[1]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# ── Row 0: persistence representations ────────────────────────────────────────
render_point_cloud(result.pts, axes[0, 0], title="Point Cloud")

# render_persistence_diagram — full persim PD; shows all dims
render_persistence_diagram(result.rips.dgms,  axes[0, 1], title="Rips PD")
render_persistence_diagram(result.alpha.dgms, axes[0, 2], title="Alpha PD")

# render_barcode — dims=None → all dims; dims=(1,) → H₁ only
render_barcode(result.rips.dgms, axes[0, 3], title="Rips Barcode  (all dims)")

# ── Row 1: remaining renderers ────────────────────────────────────────────────
# render_landscape — tent-function; dims=(1,) draws H₁ landscapes only
render_landscape(result.alpha.dgms, axes[1, 0], title="Alpha Landscape  H₁", dims=(1,))

# render_comparison_table — takes RipsResult + AlphaResult dataclasses directly
render_comparison_table(result.rips, result.alpha, axes[1, 1])

# render_voronoi_delaunay — circumsphere=True overlays circumscribed circles
render_voronoi_delaunay(result.pts, axes[1, 2], circumsphere=True)

# render_alpha_overlay — circumcenter: green=inside α, red=outside
render_alpha_overlay(result.pts, av, axes[1, 3], circumcenter=True)

plt.tight_layout()
plt.show()

# render_matching — draw bottleneck / Wasserstein matching lines between two H1 sets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
render_point_cloud(result.pts, axes[0])
render_matching(d1['d_rips'], d1['d_alpha'], d1['bn_match'], axes[1],
                title=f"H1 Bottleneck  (dist={d1['bottleneck']:.4f})")
render_matching(d1['d_rips'], d1['d_alpha'], d1['ws_match'], axes[2],
                title=f"H1 Wasserstein  (dist={d1['wasserstein']:.4f})")

# render_vr_overlay — uniform balls + edges within 2r + filled triangle cliques
render_vr_overlay(result.pts, av**0.5, axes[0])
plt.tight_layout()
plt.show()

## Layer 3 — `tda_figures`: Figure Orchestrators

Each function owns its `GridSpec` layout and calls `plt.show()`.

| Function | Description |
|---|---|
| `plot_complex_comparison(result, threshold, representations)` | 2-row Rips vs Alpha: pd / bc / pl columns |
| `plot_geometric(pts, alpha_value, ...)` | Voronoi+Delaunay (left) vs Alpha overlay (right) |
| `plot_alpha_vr_comparison(pts, alpha_value, ...)` | Alpha (left) vs VR (right) at same radius |
| `plot_distance_comparison(result)` | 2×2: PDs + H1 bottleneck/Wasserstein matchings |
| `plot_full_analysis(result, threshold, alpha_value, ...)` | Unified 5-panel figure |
| `print_distance_table(distances)` | Stdout distance summary (see Layer 1 demo above) |

In [ ]:
# plot_complex_comparison — 2-row Rips (top) vs Alpha (bottom)
# representations selects which diagram columns appear, in order
result_circle = compute_both('circle', n_points=200, noise=0.05)
plot_complex_comparison(result_circle, threshold=0.0, representations=['pd', 'bc', 'pl'])

# With threshold to suppress noise bars; circle and torus comparison
result_torus = compute_both('torus', n_points=500, noise=0.1)
plot_complex_comparison(result_torus, threshold=0.1, representations=['pd', 'bc'])

In [ ]:
# plot_geometric — Voronoi+Delaunay (left) | Alpha complex (right)
# alpha_value is in GUDHI r² units; circumsphere/circumcenter are optional overlays
pts = generate_point_cloud('circle', n_points=30, noise=0.05)
av  = auto_alpha_value(compute_alpha(pts, max_dim=1).dgms)

plot_geometric(pts, alpha_value=av, shape='circle', noise=0.05,
               circumsphere=True, circumcenter=True)

# Sweep across alpha values to watch the complex grow
for alpha_val in [0.01, 0.05, 0.15, 0.50]:
    plot_geometric(pts, alpha_value=alpha_val, shape='circle', noise=0.05,
                   circumcenter=True)

# plot_alpha_vr_comparison — Alpha (left) vs Vietoris-Rips (right) at the same r
# Alpha ⊆ VR always — the difference shows which VR simplices Alpha excludes
for alpha_val in [0.02, 0.08, 0.30]:
    plot_alpha_vr_comparison(pts, alpha_value=alpha_val,
                             shape='circle', noise=0.05, circumcenter=True)

In [ ]:
# plot_distance_comparison — 2×2 figure: Rips PD | Alpha PD | H1 bn matching | H1 ws matching
# Also prints the distance table to stdout
result_circle = compute_both('circle', n_points=200, noise=0.05)
plot_distance_comparison(result_circle)

result_torus = compute_both('torus', n_points=300, noise=0.05)
plot_distance_comparison(result_torus)

In [ ]:
# plot_full_analysis — unified 5-panel figure
#
#   [cloud]   [Rips PD]     [Alpha PD]
#   [alpha]   [Rips BC]     [Alpha BC]
#             [summary table          ]
#
# alpha_value=None auto-derives from the top-persistence H1 bar

result_circle = compute_both('circle', n_points=100, noise=0.05)
plot_full_analysis(result_circle, threshold=0.0)

# Torus with threshold to clean up noise bars
result_torus = compute_both('torus', n_points=1000, noise=0.05)
plot_full_analysis(result_torus, threshold=0.25, circumcenter=False)